# E-commerce Dashboard — Preview & Validation

Run this notebook **after** the Medallion pipeline completes (Gold tables built).

**What it does:**
1. Confirms Gold tables exist and have data
2. Runs all dashboard SQL queries and displays results
3. Prints next steps to build the Lakeview / SQL Dashboard UI

**Prerequisite:** Run `notebooks/databricks_serverless_pipeline.ipynb` first (or ensure Gold tables exist in schema `ecommerce`).

**SQL Dashboard setup:** See `src/dashboard/DASHBOARD_GUIDE.md` and copy queries from `src/dashboard/dashboard_queries.sql`.

In [ ]:
# =============================================================================
# CELL 1 — Configuration
# =============================================================================
dbutils.widgets.text("schema_name", "ecommerce", "Hive schema with Gold tables")
dbutils.widgets.text("product_category", "ALL", "Filter: ALL or category name")
dbutils.widgets.text("customer_segment", "ALL", "Filter: ALL, Premium, Standard, Basic")

SCHEMA_NAME = dbutils.widgets.get("schema_name").strip() or "ecommerce"
PRODUCT_CATEGORY = dbutils.widgets.get("product_category").strip() or "ALL"
CUSTOMER_SEGMENT = dbutils.widgets.get("customer_segment").strip() or "ALL"

spark.sql(f"USE {SCHEMA_NAME}")

print(f"schema={SCHEMA_NAME}")
print(f"product_category={PRODUCT_CATEGORY}")
print(f"customer_segment={CUSTOMER_SEGMENT}")

In [ ]:
# =============================================================================
# CELL 2 — Validate Gold tables
# =============================================================================
GOLD_TABLES = (
    "gold_sales_by_product",
    "gold_revenue_by_customer",
    "gold_customer_segmentation",
    "gold_daily_weekly_trends",
)

missing = [t for t in GOLD_TABLES if not spark.catalog.tableExists(t)]
if missing:
    raise RuntimeError(
        f"Missing Gold tables in schema '{SCHEMA_NAME}': {missing}. "
        "Run the pipeline notebook first."
    )

display(
    spark.sql("""
        SELECT 'gold_sales_by_product' AS table_name, COUNT(*) AS row_count FROM gold_sales_by_product
        UNION ALL SELECT 'gold_revenue_by_customer', COUNT(*) FROM gold_revenue_by_customer
        UNION ALL SELECT 'gold_customer_segmentation', COUNT(*) FROM gold_customer_segmentation
        UNION ALL SELECT 'gold_daily_weekly_trends', COUNT(*) FROM gold_daily_weekly_trends
    """)
)
print("Gold tables OK")

In [ ]:
# =============================================================================
# CELL 3 — Visualization 1: Top 10 Products by Revenue (Bar chart)
# =============================================================================
pc = PRODUCT_CATEGORY.replace("'", "''")
display(
    spark.sql(f"""
        WITH dashboard_filters AS (
            SELECT '{pc}' AS product_category
        )
        SELECT
            p.product_name,
            p.category,
            p.total_orders,
            p.total_revenue,
            p.avg_order_value
        FROM gold_sales_by_product AS p
        CROSS JOIN dashboard_filters AS f
        WHERE f.product_category = 'ALL' OR p.category = f.product_category
        ORDER BY p.total_revenue DESC
        LIMIT 10
    """)
)

In [ ]:
# =============================================================================
# CELL 4 — Visualization 2: Customer Revenue Distribution (Histogram)
# =============================================================================
cs = CUSTOMER_SEGMENT.replace("'", "''")
display(
    spark.sql(f"""
        WITH dashboard_filters AS (
            SELECT '{cs}' AS customer_segment
        )
        SELECT revenue_bucket, COUNT(*) AS customer_count
        FROM (
            SELECT
                CASE
                    WHEN c.total_revenue = 0 THEN '$0'
                    WHEN c.total_revenue <= 100 THEN '$1 - $100'
                    WHEN c.total_revenue <= 500 THEN '$101 - $500'
                    WHEN c.total_revenue <= 1000 THEN '$501 - $1,000'
                    WHEN c.total_revenue <= 5000 THEN '$1,001 - $5,000'
                    ELSE '$5,000+'
                END AS revenue_bucket,
                CASE
                    WHEN c.total_revenue = 0 THEN 1
                    WHEN c.total_revenue <= 100 THEN 2
                    WHEN c.total_revenue <= 500 THEN 3
                    WHEN c.total_revenue <= 1000 THEN 4
                    WHEN c.total_revenue <= 5000 THEN 5
                    ELSE 6
                END AS bucket_sort_order
            FROM gold_revenue_by_customer AS c
            CROSS JOIN dashboard_filters AS f
            WHERE f.customer_segment = 'ALL' OR c.customer_segment = f.customer_segment
        ) AS bucketed
        GROUP BY revenue_bucket, bucket_sort_order
        ORDER BY bucket_sort_order
    """)
)

In [ ]:
# =============================================================================
# CELL 5 — Visualization 3: Customer Segmentation (Pie chart)
# =============================================================================
display(
    spark.sql("""
        SELECT segment_type, customer_count, total_revenue, avg_revenue
        FROM gold_customer_segmentation
        ORDER BY
            CASE segment_type
                WHEN 'High-Value' THEN 1
                WHEN 'Repeat' THEN 2
                WHEN 'One-Time' THEN 3
                WHEN 'Inactive' THEN 4
                ELSE 5
            END
    """)
)

In [ ]:
# =============================================================================
# CELL 6 — Optional: Daily Revenue Trend (Line chart)
# =============================================================================
display(
    spark.sql("""
        WITH trend_bounds AS (
            SELECT MIN(period_start_date) AS start_date, MAX(period_start_date) AS end_date
            FROM gold_daily_weekly_trends
            WHERE period_grain = 'DAY'
        )
        SELECT t.period_start_date, t.order_count, t.total_revenue
        FROM gold_daily_weekly_trends AS t
        CROSS JOIN trend_bounds AS f
        WHERE t.period_grain = 'DAY'
          AND t.period_start_date >= f.start_date
          AND t.period_start_date <= f.end_date
        ORDER BY t.period_start_date
    """)
)

## Next: Create the SQL Dashboard (UI)

This notebook **previews** data. To build the shareable dashboard:

1. **SQL** → **Queries** → **Create query** — paste each block from `src/dashboard/dashboard_queries.sql`, save 3 queries.
2. **SQL** → **Dashboards** → **Create dashboard** → **Add visualization** for each saved query.
3. Configure charts:
   - **Top 10 Products** — Bar, X=`product_name`, Y=`total_revenue`
   - **Revenue Distribution** — Bar, X=`revenue_bucket`, Y=`customer_count`
   - **Segmentation** — Pie, slice=`segment_type`, value=`customer_count`
4. Save the dashboard.

Full guide: `src/dashboard/DASHBOARD_GUIDE.md`